In [0]:

from pyspark.sql.functions import col

claims_path = "/Volumes/workspace/default/insurance_landing/claims_core.csv"

claims = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(claims_path)
)

print("Total rows:", claims.count())
print("Columns:", claims.columns)

display(claims.limit(5))


Total rows: 15420
Columns: ['claim_id', 'policy_number', 'claim_year', 'claim_month', 'claim_week_of_month', 'accident_day_of_week', 'claimed_day_of_week', 'claimed_month', 'claimed_week_of_month', 'accident_area', 'fault', 'policy_type', 'base_policy', 'deductible', 'vehicle_category', 'vehicle_make', 'vehicle_price', 'vehicle_age', 'customer_age', 'customer_sex', 'marital_status', 'driver_rating', 'police_report_filed', 'witness_present', 'agent_type', 'agent_id', 'number_of_supplements', 'address_change_claim', 'number_of_cars', 'days_policy_accident', 'days_policy_claim', 'past_number_of_claims', 'age_of_policyholder', 'fraud_label']


claim_id,policy_number,claim_year,claim_month,claim_week_of_month,accident_day_of_week,claimed_day_of_week,claimed_month,claimed_week_of_month,accident_area,fault,policy_type,base_policy,deductible,vehicle_category,vehicle_make,vehicle_price,vehicle_age,customer_age,customer_sex,marital_status,driver_rating,police_report_filed,witness_present,agent_type,agent_id,number_of_supplements,address_change_claim,number_of_cars,days_policy_accident,days_policy_claim,past_number_of_claims,age_of_policyholder,fraud_label
CLM000001,1,1994,Dec,5,Wednesday,Tuesday,Jan,1,Urban,Policy Holder,Sport - Liability,Liability,300,Sport,Honda,more than 69000,3 years,21,Female,Single,1,No,No,External,12,none,1 year,3 to 4,more than 30,more than 30,none,26 to 30,0
CLM000002,2,1994,Jan,3,Wednesday,Monday,Jan,4,Urban,Policy Holder,Sport - Collision,Collision,400,Sport,Honda,more than 69000,6 years,34,Male,Single,4,Yes,No,External,15,none,no change,1 vehicle,more than 30,more than 30,none,31 to 35,0
CLM000003,3,1994,Oct,5,Friday,Thursday,Nov,2,Urban,Policy Holder,Sport - Collision,Collision,400,Sport,Honda,more than 69000,7 years,47,Male,Married,3,No,No,External,7,none,no change,1 vehicle,more than 30,more than 30,1,41 to 50,0
CLM000004,4,1994,Jun,2,Saturday,Friday,Jul,1,Rural,Third Party,Sedan - Liability,Liability,400,Sport,Toyota,20000 to 29000,more than 7,65,Male,Married,2,Yes,No,External,4,more than 5,no change,1 vehicle,more than 30,more than 30,1,51 to 65,0
CLM000005,5,1994,Jan,5,Monday,Tuesday,Feb,2,Urban,Third Party,Sport - Collision,Collision,400,Sport,Honda,more than 69000,5 years,27,Female,Single,1,No,No,External,3,none,no change,1 vehicle,more than 30,more than 30,none,31 to 35,0


In [0]:

from pyspark.sql.functions import col, lit

table_name = "workspace.default.insurance_claims_delta_demo"

# Check that claim IDs are valid and unique
if claims.filter(col("claim_id").isNull()).count() > 0:
    raise ValueError("Missing claim IDs found")

if claims.count() != claims.select("claim_id").distinct().count():
    raise ValueError("Duplicate claim IDs found")

# Create a separate demonstration table
delta_data = claims.withColumn(
    "delta_demo_status", lit("ORIGINAL")
)

delta_data.write.format("delta").mode("overwrite").saveAsTable(table_name)

print("Delta table created:", table_name)
print("Rows:", spark.table(table_name).count())

display(spark.table(table_name).limit(5))


Delta table created: workspace.default.insurance_claims_delta_demo
Rows: 15420


claim_id,policy_number,claim_year,claim_month,claim_week_of_month,accident_day_of_week,claimed_day_of_week,claimed_month,claimed_week_of_month,accident_area,fault,policy_type,base_policy,deductible,vehicle_category,vehicle_make,vehicle_price,vehicle_age,customer_age,customer_sex,marital_status,driver_rating,police_report_filed,witness_present,agent_type,agent_id,number_of_supplements,address_change_claim,number_of_cars,days_policy_accident,days_policy_claim,past_number_of_claims,age_of_policyholder,fraud_label,delta_demo_status
CLM000001,1,1994,Dec,5,Wednesday,Tuesday,Jan,1,Urban,Policy Holder,Sport - Liability,Liability,300,Sport,Honda,more than 69000,3 years,21,Female,Single,1,No,No,External,12,none,1 year,3 to 4,more than 30,more than 30,none,26 to 30,0,ORIGINAL
CLM000002,2,1994,Jan,3,Wednesday,Monday,Jan,4,Urban,Policy Holder,Sport - Collision,Collision,400,Sport,Honda,more than 69000,6 years,34,Male,Single,4,Yes,No,External,15,none,no change,1 vehicle,more than 30,more than 30,none,31 to 35,0,ORIGINAL
CLM000003,3,1994,Oct,5,Friday,Thursday,Nov,2,Urban,Policy Holder,Sport - Collision,Collision,400,Sport,Honda,more than 69000,7 years,47,Male,Married,3,No,No,External,7,none,no change,1 vehicle,more than 30,more than 30,1,41 to 50,0,ORIGINAL
CLM000004,4,1994,Jun,2,Saturday,Friday,Jul,1,Rural,Third Party,Sedan - Liability,Liability,400,Sport,Toyota,20000 to 29000,more than 7,65,Male,Married,2,Yes,No,External,4,more than 5,no change,1 vehicle,more than 30,more than 30,1,51 to 65,0,ORIGINAL
CLM000005,5,1994,Jan,5,Monday,Tuesday,Feb,2,Urban,Third Party,Sport - Collision,Collision,400,Sport,Honda,more than 69000,5 years,27,Female,Single,1,No,No,External,3,none,no change,1 vehicle,more than 30,more than 30,none,31 to 35,0,ORIGINAL


In [0]:

from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit

table_name = "workspace.default.insurance_claims_delta_demo"

# Find one existing claim
first_id = (
    spark.table(table_name)
    .orderBy("claim_id")
    .first()["claim_id"]
)

# Prepare one update and one synthetic insert
updates = spark.createDataFrame(
    [
        (first_id, "UPDATED_BY_MERGE"),
        ("DELTA_DEMO_INSERT_001", "SYNTHETIC_TEST")
    ],
    ["claim_id", "delta_demo_status"]
)

# Perform MERGE
target = DeltaTable.forName(spark, table_name)

(
    target.alias("target")
    .merge(
        updates.alias("source"),
        "target.claim_id = source.claim_id"
    )
    .whenMatchedUpdate(
        set={"delta_demo_status": "source.delta_demo_status"}
    )
    .whenNotMatchedInsert(
        values={
            "claim_id": "source.claim_id",
            "delta_demo_status": "source.delta_demo_status"
        }
    )
    .execute()
)

print("MERGE completed successfully")

# Verify updated and inserted rows
display(
    spark.table(table_name)
    .filter(
        col("claim_id").isin(
            first_id,
            "DELTA_DEMO_INSERT_001"
        )
    )
    .select("claim_id", "delta_demo_status")
)

# Show transaction history
print("Delta Lake transaction history:")
display(spark.sql(f"DESCRIBE HISTORY {table_name}"))

# Verify time travel
print("Original version (version 0):")
display(spark.sql(f"""
    SELECT COUNT(*) AS original_rows
    FROM {table_name} VERSION AS OF 0
"""))

print("Current version:")
display(spark.sql(f"""
    SELECT COUNT(*) AS current_rows
    FROM {table_name}
"""))

print("Delta Lake checks completed.")


MERGE completed successfully


claim_id,delta_demo_status
CLM000001,UPDATED_BY_MERGE
DELTA_DEMO_INSERT_001,SYNTHETIC_TEST


Delta Lake transaction history:


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-22T06:37:56.000Z,72642014943382,cs22b1024lisha@gmail.com,MERGE,"Map(predicate -> [""(claim_id#13728 = claim_id#13771)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1962323354776443),29b1888c-dd59-418b-b6bb-860e8dee912b,1022-063316-y7f8befd-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 16642, numTargetBytesRemoved -> 16642, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 4104, materializeSourceTimeMs -> 164, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1509, numTargetRowsUpdated -> 2, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2388)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
1,2026-10-22T06:36:28.000Z,72642014943382,cs22b1024lisha@gmail.com,MERGE,"Map(predicate -> [""(claim_id#11898 = claim_id#11941)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1962323354776443),9ca09a4f-0b39-4c15-9601-55d1818a43ab,1022-063316-y7f8befd-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 16642, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 8238, materializeSourceTimeMs -> 460, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2950, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4683)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13
0,2026-10-22T06:35:13.000Z,72642014943382,cs22b1024lisha@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1962323354776443),f7f43968-9055-4a73-9089-2970a524021a,1022-063316-y7f8befd-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 15420, numOutputBytes -> 129665)",null,Databricks-Runtime/19.11.x-aarch64-photon-scala2.13


Original version (version 0):


original_rows
15420


Current version:


current_rows
15421


Delta Lake checks completed.
